In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

OUTPUT_DIR = "/kaggle/working/rubert-tapt"
MODEL_NAME = "DeepPavlov/rubert-base-cased-conversational"
DATA_FILE = "/kaggle/input/datasets/timurx/tapt-corpus-cleaned/tapt_corpus.jsonl" 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

SPECIAL_TOKENS = ["[MEDIA_URL]", "[URL]", "[EMAIL]", "[PHONE]"]

num_added_toks = tokenizer.add_tokens(SPECIAL_TOKENS, special_tokens=True)
logger.info(f"Добавлено новых спец-токенов: {num_added_toks}")


model.resize_token_embeddings(len(tokenizer))
logger.info(f"Новый размер словаря: {len(tokenizer)}")

dataset = load_dataset("json", data_files=DATA_FILE)

split_dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=512
    )

logger.info("Токенизация текстов")

tokenized_datasets = split_dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=dataset["train"].column_names 
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

model.config.use_cache = False 
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=3e-5,
    num_train_epochs=10, 
    
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    
    warmup_steps=500,
    weight_decay=0.01,
    
    eval_strategy="steps",       
    eval_steps=1000,
    save_strategy="steps",       
    save_steps=1000,      
    load_best_model_at_end=True,  
    metric_for_best_model="loss", 
    greater_is_better=False,
    
    logging_steps=100,
    save_total_limit=2,
    
    fp16=torch.cuda.is_available(), 
    report_to="none" 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

logger.info("Старт TAPT (MLM)")
trainer.train()

logger.info(f"Сохранение лучшей адаптированной модели в {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased-conversational/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased-conversational/645946ce91842a52eaacb2705c77e59194145ffa/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased-conversational/645946ce91842a52eaacb2705c77e59194145ffa/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased-conversational/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased-conversational/645946ce91842a52eaacb2705c77e59194145ffa/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased-conversational/645946ce91842a52eaacb2705c77e59194145ffa/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/DeepPavlov/rubert-base-cased-conversational/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/DeepPavlov/rubert-base-cased-conversational/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased-conversational/resolve/main/vocab.txt "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased-conversational/645946ce91842a52eaacb2705c77e59194145ffa/vocab.txt "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased-conversational/645946ce91842a52eaacb2705c77e59194145ffa/vocab.txt "HTTP/1.1 200 OK"


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [3]:
import os
from IPython.display import FileLink

folder_to_zip = "/kaggle/working/rubert-tapt"
output_zip = "/kaggle/working/rubert-tapt.zip"

print("Архивируем веса... Это может занять пару минут.")
os.system(f"zip -qr {output_zip} {folder_to_zip}")
print("Архивация завершена!")


Архивируем веса... Это может занять пару минут.
Архивация завершена!


In [4]:
FileLink(r'rubert-tapt.zip')

/kaggle/working/rubert-tapt.zip